# FER-CE Vision-LLM: BLIP-2 for Emotion Explanation

This notebook implements a Vision-Language model using BLIP-2 to classify compound emotions and generate textual explanations based on facial cues.

In [ ]:
import os
import sys
import torch
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Add src to path
sys.path.append(os.path.abspath('../src'))
from dataset import RAFCEDataset

## 1. Mapping & Config

In [ ]:
EMO_MAP = {
    0: "Happily surprised",
    1: "Happily disgusted",
    2: "Sadly fearful",
    3: "Sadly angry",
    4: "Sadly surprised",
    5: "Sadly disgusted",
    6: "Fearfully angry",
    7: "Fearfully surprised",
    8: "Fearfully disgusted",
    9: "Angrily surprised",
    10: "Angrily disgusted",
    11: "Disgustedly surprised",
    12: "Happily fearful",
    13: "Happily sad"
}

IMG_DIR = '../../aligned'
LABEL_FILE = '../../RAFCE_emolabel.txt'
PARTITION_FILE = '../../RAFCE_partition.txt'
OUTPUT_DIR = '../outputs/vision_llm'
MODEL_ID = "Salesforce/blip2-opt-2.7b"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Load Model & Processor

In [ ]:
processor = Blip2Processor.from_pretrained(MODEL_ID)
model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_ID, 
    load_in_8bit=True if torch.cuda.is_available() else False,
    device_map="auto" if torch.cuda.is_available() else None
)

if not torch.cuda.is_available():
    model = model.to(DEVICE)

# LoRA Setup
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, config)

## 3. Data Loading for VLM

In [ ]:
class BLIPDataset(RAFCEDataset):
    def __getitem__(self, idx):
        # Get image paths and label IDs
        img_name = self.data.iloc[idx, 0]
        base_name = img_name.split('.')[0]
        full_img_name = f"{base_name}_aligned.jpg"
        img_path = os.path.join(self.root_dir, full_img_name)
        
        image = Image.open(img_path).convert('RGB')
        label_id = int(self.data.iloc[idx, 1])
        emotion = EMO_MAP[label_id]
        
        # Formalize prompt and answer
        prompt = "Question: Describe the emotional state of this person and explain why. Answer:"
        target_text = f"The person seems {emotion}: based on facial muscle movements."
        
        return image, prompt, target_text

dataset = BLIPDataset(IMG_DIR, LABEL_FILE, PARTITION_FILE, split=2) # Using test split for demo
image, prompt, target = dataset[0]
plt.imshow(image)
print(f"Target: {target}")

## 4. Zero-Shot Inference

In [ ]:
def infer(image, prompt):
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(DEVICE, torch.float16 if torch.cuda.is_available() else torch.float32)
    generated_ids = model.generate(**inputs)
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return generated_text

print("Inference:", infer(image, prompt))

## 5. Training / Fine-Tuning Setup

Note: Due to memory constraints, training BLIP-2 usually requires a high-end GPU (A10/A100). LoRA allows fine-tuning on consumer GPUs.

In [ ]:
# Placeholder for training loop - similar to baseline but using language loss
print("Training logic would go here using HuggingFace Trainer or custom loop.")